In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error,mean_squared_error

In [2]:
df = pd.read_csv('cleaned_engineered.csv')

In [3]:
df.head()

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4


In [20]:
skew_num = ['Study_Hours']
other_num = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ord_cat = ['Stress_Level','Academic_Level']
ohe_cat = ['Gender','Country','Most_Used_Platform','Purpose_Of_Use']
cols = skew_num+other_num+ord_cat+ohe_cat
X = df[cols]
y = df['Mental_Health_Score']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [33]:
#1. Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

#3. Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High'],['High School','Undergraduate','Graduate',]]))
])

#4. Nominal Features
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore",drop='first'))
])


preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skew_num),
    ("Plain_Numeric",plain_numeric_pipeline, other_num ),
    ('Ordinal', ordinal_pipeline, ord_cat),
    ('Normal', nominal_pipeline, ohe_cat)
])


In [34]:
lr_pipe = Pipeline([['preprocess',preprocessor],['lr',LinearRegression()]])

In [35]:
lr_pipe.fit(X_train,y_train)

,steps,"[('preprocess', ...), ['lr', LinearRegression()]]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [36]:
y_pred_test = lr_pipe.predict(X_test)
y_pred_train = lr_pipe.predict(X_train)

In [38]:
print("Test r2",r2_score(y_test,y_pred_test))
print("Train r2",r2_score(y_train,y_pred_train))
print ("Test MAE", mean_absolute_error(y_test,y_pred_test))
print("Test RMSE",root_mean_squared_error(y_test,y_pred_test))

Test r2 0.7844265660459843
Train r2 0.772664125468838
Test MAE 0.4865808387688088
Test RMSE 0.620357520607214


In [41]:
scores = cross_val_score(lr_pipe,X,y,cv=5,scoring='r2')

In [42]:
scores

array([0.77010146, 0.79641672, 0.76327384, 0.76462012, 0.76291879])

In [43]:
scores.mean()

np.float64(0.771466184929004)